# Análise de Vendas — Superstore

**Objetivo:** Explorar o dataset Superstore para identificar padrões de vendas, sub-categorias mais lucrativas, impacto de descontos e sazonalidade.

**Perguntas que vamos responder:**
- Quais sub-categorias vendem mais (e quais dão prejuízo)?
- Descontos altos realmente destroem o lucro?
- Como as vendas evoluem ao longo do tempo?
- O que mais se correlaciona com o lucro?

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

df = pd.read_csv(
    'data/Sample - Superstore.csv',
    encoding='latin1'
)
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date']  = pd.to_datetime(df['Ship Date'])

print(f'Linhas: {df.shape[0]:,} | Colunas: {df.shape[1]}')
df.head(3)

## 1. Visão geral dos dados

In [ ]:
resumo = pd.DataFrame({
    'Métrica': ['Total de pedidos', 'Clientes únicos', 'Produtos únicos',
                'Vendas totais', 'Lucro total', 'Margem média'],
    'Valor': [
        f"{df['Order ID'].nunique():,}",
        f"{df['Customer ID'].nunique():,}",
        f"{df['Product ID'].nunique():,}",
        f"$ {df['Sales'].sum():,.0f}",
        f"$ {df['Profit'].sum():,.0f}",
        f"{df['Profit'].sum() / df['Sales'].sum() * 100:.1f}%"
    ]
})
resumo

## 2. Vendas e lucro por sub-categoria

Identificamos quais sub-categorias geram mais receita e quais estão com margem negativa.

In [ ]:
sub = (
    df.groupby('Sub-Category')[['Sales', 'Profit']]
    .sum()
    .sort_values('Sales', ascending=True)
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

cores_vendas = ['#1D9E75' if v >= 0 else '#D85A30' for v in sub['Sales']]
axes[0].barh(sub['Sub-Category'], sub['Sales'], color=cores_vendas, edgecolor='none')
axes[0].axvline(sub['Sales'].mean(), color='gray', linestyle='--', linewidth=1, label=f"Média: ${sub['Sales'].mean():,.0f}")
axes[0].set_title('Vendas totais por sub-categoria', fontsize=13, fontweight='bold', pad=12)
axes[0].set_xlabel('Vendas (USD)')
axes[0].legend(fontsize=10)
axes[0].xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))

sub_lucro = sub.sort_values('Profit', ascending=True)
cores_lucro = ['#D85A30' if v < 0 else '#1D9E75' for v in sub_lucro['Profit']]
axes[1].barh(sub_lucro['Sub-Category'], sub_lucro['Profit'], color=cores_lucro, edgecolor='none')
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Lucro total por sub-categoria', fontsize=13, fontweight='bold', pad=12)
axes[1].set_xlabel('Lucro (USD)')
axes[1].xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))

plt.suptitle('Sub-categorias: vendas vs. lucro', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('images/01_subcategoria.png', dpi=150, bbox_inches='tight')
plt.show()

prejuizo = sub_lucro[sub_lucro['Profit'] < 0]['Sub-Category'].tolist()
print(f'Sub-categorias com prejuízo: {prejuizo}')

## 3. Desconto vs. Lucro — onde perdemos dinheiro?

Descontos acima de 20% tendem a virar prejuízo. O gráfico abaixo mostra essa relação para cada venda.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

scatter = ax.scatter(
    df['Discount'],
    df['Profit'],
    c=df['Profit'],
    cmap='RdYlGn',
    vmin=-500, vmax=500,
    alpha=0.5,
    s=30,
    edgecolors='none'
)

plt.colorbar(scatter, ax=ax, label='Lucro por venda (USD)', shrink=0.8)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--', label='Break-even')
ax.axvline(0.2, color='#D85A30', linewidth=1.2, linestyle=':', label='20% de desconto')

perdas = df[df['Discount'] >= 0.2]
pct_prej = (perdas['Profit'] < 0).mean() * 100
ax.text(0.22, ax.get_ylim()[0] * 0.8,
        f'{pct_prej:.0f}% das vendas\ncom desc. ≥20%\ndão prejuízo',
        fontsize=10, color='#D85A30')

ax.set_xlabel('Desconto aplicado', fontsize=12)
ax.set_ylabel('Lucro (USD)', fontsize=12)
ax.set_title('Desconto vs. Lucro — onde perdemos dinheiro?', fontsize=13, fontweight='bold', pad=12)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('images/02_desconto_lucro.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Sazonalidade — evolução de vendas ao longo do tempo

Analisamos as vendas mensais para identificar picos e tendência de crescimento.

In [ ]:
df['YearMonth'] = df['Order Date'].dt.to_period('M')
temporal = (
    df.groupby('YearMonth')['Sales']
    .sum()
    .reset_index()
)
temporal['YearMonth'] = temporal['YearMonth'].astype(str)
x = range(len(temporal))

fig, ax = plt.subplots(figsize=(13, 5))

ax.fill_between(x, temporal['Sales'], alpha=0.12, color='#1D9E75')
ax.plot(x, temporal['Sales'], color='#1D9E75', linewidth=2)

idx_max = temporal['Sales'].idxmax()
ax.annotate(
    f"Pico: ${temporal['Sales'][idx_max]:,.0f}",
    xy=(idx_max, temporal['Sales'][idx_max]),
    xytext=(idx_max - 4, temporal['Sales'][idx_max] * 1.05),
    arrowprops=dict(arrowstyle='->', color='#085041'),
    fontsize=10, color='#085041'
)

step = 3
ax.set_xticks(list(x)[::step])
ax.set_xticklabels(temporal['YearMonth'].iloc[::step], rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Vendas (USD)')
ax.set_title('Evolução das vendas mensais', fontsize=13, fontweight='bold', pad=12)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))

plt.tight_layout()
plt.savefig('images/03_sazonalidade.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Mapa de calor — correlação entre variáveis

Verificamos o grau de correlação entre vendas, quantidade, desconto e lucro.

In [ ]:
numericas = df[['Sales', 'Quantity', 'Discount', 'Profit']]
corr = numericas.corr()

fig, ax = plt.subplots(figsize=(7, 5))
mask = pd.DataFrame(False, index=corr.index, columns=corr.columns)

sns.heatmap(
    corr,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.5,
    linecolor='white',
    ax=ax,
    annot_kws={'size': 12, 'weight': 'bold'},
    square=True
)

ax.set_title('Correlação entre variáveis numéricas', fontsize=13, fontweight='bold', pad=12)
ax.tick_params(labelsize=11)
plt.tight_layout()
plt.savefig('images/04_correlacao.png', dpi=150, bbox_inches='tight')
plt.show()

print('Correlação com Profit:')
print(corr['Profit'].drop('Profit').sort_values())

## Conclusões

- **Sub-categorias lucrativas:** Technology lidera em vendas; Tables e Bookcases apresentam prejuízo sistemático.
- **Desconto é armadilha:** Vendas com desconto acima de 20% têm alta proporção de resultado negativo.
- **Sazonalidade clara:** Pico de vendas no Q4 (novembro/dezembro) em todos os anos analisados.
- **Correlação:** Desconto tem correlação negativa com lucro — confirma o insight anterior.

---
*Análise realizada com Python (pandas, matplotlib, seaborn)*